<a href="https://colab.research.google.com/github/carolshayle/Python/blob/main/Yolov8_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""YOLOv8 Building Footprint Segmentation Pipeline for Aerial Imagery

This script provides a complete pipeline for training YOLOv8 segmentation models
on high-resolution aerial imagery to detect building footprints.

Key features:
- Handles large TIFF images and shapefiles
- Automatic tiling with configurable size and overlap
- Robust CUDA memory management with fallbacks
- Mixed precision training and checkpointing
- Visualizations and progress monitoring

Variables to configure:
- TILE_SIZE: Size of image tiles (adjust based on GPU memory)
- OVERLAP: Overlap between tiles to avoid edge artifacts
- BATCH_SIZE: Start with 8-16, will auto-reduce if OOM
- MODEL_VARIANT: 'yolov8n-seg' or 'yolov8s-seg'
- EPOCHS: Number of training epochs
- TRAIN_RATIO: Train/validation split ratio

Usage:
1. Run in Google Colab with GPU runtime
2. Upload files when prompted: Train.tif, Test.tif, and shapefile components
3. Monitor training progress and visualizations
"""

# Install dependencies
!pip install ultralytics rasterio geopandas shapely pycocotools matplotlib tqdm -q

import os
import cv2
import torch
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import Polygon, box
from pycocotools import mask as coco_mask
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import shutil
import tempfile
from pathlib import Path

# Configuration - ADJUST THESE PARAMETERS AS NEEDED
TILE_SIZE = 640  # Image tile size (smaller = less memory, but more context loss)
OVERLAP = 128    # Overlap between tiles to avoid edge artifacts
BATCH_SIZE = 8   # Start with this batch size, will auto-reduce if OOM
MODEL_VARIANT = 'yolov8s-seg'  # 'yolov8n-seg' (faster) or 'yolov8s-seg' (more accurate)
EPOCHS = 50      # Number of training epochs
TRAIN_RATIO = 0.8  # Train/validation split ratio
MIN_BUILDING_PIXELS = 50  # Minimum building pixels to keep a tile
MAX_EMPTY_TILES = 0.2     # Maximum fraction of empty tiles to keep for balance

# Create directories
os.makedirs('data/train/images', exist_ok=True)
os.makedirs('data/train/labels', exist_ok=True)
os.makedirs('data/val/images', exist_ok=True)
os.makedirs('data/val/labels', exist_ok=True)
os.makedirs('data/test/images', exist_ok=True)
os.makedirs('test_predictions', exist_ok=True)

# Clean up CUDA memory
def cleanup_memory():
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

cleanup_memory()

# File upload
print("Please upload your files:")
print("1. Train.tif - Training aerial imagery")
print("2. Test.tif - Test aerial imagery")
print("3. Shapefile components (.shp, .shx, .dbf, .prj)")

from google.colab import files
uploaded = files.upload()

# Extract shapefile name (without extension)
shapefile_name = None
for filename in uploaded.keys():
    if filename.endswith('.shp'):
        shapefile_name = filename[:-4]
        break

if shapefile_name is None:
    raise ValueError("No .shp file found in uploads!")

# Save uploaded files
for filename, content in uploaded.items():
    with open(filename, 'wb') as f:
        f.write(content)

# Load training image and shapefile
print("Loading training data...")
with rasterio.open('Train.tif') as src:
    train_img = src.read()
    train_transform = src.transform
    train_crs = src.crs

# Read shapefile and filter for buildings (Type = 1)
gdf = gpd.read_file(shapefile_name + '.shp')
building_gdf = gdf[gdf['Type'] == 1].copy()

if building_gdf.crs != train_crs:
    building_gdf = building_gdf.to_crs(train_crs)

print(f"Found {len(building_gdf)} building polygons")

# Function to create tiles from image
def create_tiles(image, transform, tile_size, overlap):
    """Create tiles from large image with overlap"""
    height, width = image.shape[1], image.shape[2]
    tiles = []
    tile_coords = []

    for y in range(0, height, tile_size - overlap):
        for x in range(0, width, tile_size - overlap):
            # Calculate tile bounds
            y_end = min(y + tile_size, height)
            x_end = min(x + tile_size, width)

            # Extract tile
            tile = image[:, y:y_end, x:x_end]

            # Pad if necessary
            if tile.shape[1] < tile_size or tile.shape[2] < tile_size:
                pad_y = tile_size - tile.shape[1]
                pad_x = tile_size - tile.shape[2]
                tile = np.pad(tile, ((0, 0), (0, pad_y), (0, pad_x)),
                             mode='constant', constant_values=0)

            tiles.append(tile)
            tile_coords.append((x, y, x_end, y_end))

    return tiles, tile_coords

# Function to rasterize polygons for a tile
def rasterize_polygons_for_tile(polygons, tile_bounds, tile_size, transform):
    """Create segmentation mask for a tile"""
    mask = np.zeros((tile_size, tile_size), dtype=np.uint8)

    # Create tile polygon
    tile_poly = box(*tile_bounds)

    for polygon in polygons:
        # Check if polygon intersects with tile
        if tile_poly.intersects(polygon):
            # Get intersection
            intersection = tile_poly.intersection(polygon)

            if not intersection.is_empty:
                # Convert to pixel coordinates
                if intersection.geom_type == 'Polygon':
                    intersections = [intersection]
                else:
                    intersections = list(intersection.geoms)

                for inter in intersections:
                    # Convert geometry to pixel coordinates
                    coords = list(inter.exterior.coords)
                    pixel_coords = []

                    for lon, lat in coords:
                        # Transform to pixel coordinates relative to tile
                        col, row = ~transform * (lon, lat)
                        col -= tile_bounds[0]  # Adjust to tile coordinates
                        row -= tile_bounds[1]
                        pixel_coords.append([col, row])

                    # Convert to integer coordinates and draw polygon
                    pts = np.array(pixel_coords, np.int32)
                    pts = pts.reshape((-1, 1, 2))
                    cv2.fillPoly(mask, [pts], 1)

    return mask

# Function to convert mask to YOLO segmentation format
def mask_to_yolo_segmentation(mask):
    """Convert binary mask to YOLO segmentation format"""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    segments = []

    for contour in contours:
        if contour.shape[0] >= 3:  # Need at least 3 points for a polygon
            # Normalize coordinates to 0-1
            contour = contour.squeeze().astype(np.float32)
            contour[:, 0] /= mask.shape[1]  # x divided by width
            contour[:, 1] /= mask.shape[0]  # y divided by height

            # Flatten and format
            segment = contour.flatten().tolist()
            segments.append(segment)

    return segments

# Create tiles and masks
print("Creating training tiles and masks...")
train_tiles, train_coords = create_tiles(train_img, train_transform, TILE_SIZE, OVERLAP)

# Prepare training data
train_data = []
empty_tiles = 0
total_tiles = len(train_tiles)

for i, (tile, bounds) in tqdm(enumerate(zip(train_tiles, train_coords)), total=total_tiles):
    # Convert bounds to geographic coordinates
    x1, y1, x2, y2 = bounds
    geo_bounds = (
        train_transform * (x1, y1),  # top-left
        train_transform * (x2, y1),  # top-right
        train_transform * (x2, y2),  # bottom-right
        train_transform * (x1, y2)   # bottom-left
    )

    # Create polygon for tile bounds
    tile_poly = box(geo_bounds[0][0], geo_bounds[0][1],
                   geo_bounds[2][0], geo_bounds[2][1])

    # Find polygons that intersect with this tile
    intersecting_polygons = []
    for _, row in building_gdf.iterrows():
        if tile_poly.intersects(row.geometry):
            intersecting_polygons.append(row.geometry)

    # Create mask
    mask = rasterize_polygons_for_tile(intersecting_polygons,
                                     (x1, y1, x2, y2),
                                     TILE_SIZE, train_transform)

    # Count building pixels
    building_pixels = np.sum(mask > 0)

    # Skip if no buildings (but keep some for balance)
    if building_pixels < MIN_BUILDING_PIXELS:
        empty_tiles += 1
        if empty_tiles / total_tiles > MAX_EMPTY_TILES:
            continue

    # Save tile image
    tile_img = np.moveaxis(tile, 0, -1)  # CHW to HWC
    cv2.imwrite(f'data/train/images/tile_{i:04d}.png', tile_img)

    # Convert mask to YOLO format and save
    segments = mask_to_yolo_segmentation(mask)
    if segments:
        with open(f'data/train/labels/tile_{i:04d}.txt', 'w') as f:
            for segment in segments:
                f.write(f"0 {' '.join(map(str, segment))}\n")

    train_data.append((f'tile_{i:04d}.png', building_pixels))

print(f"Created {len(train_data)} training tiles ({empty_tiles} empty tiles discarded)")

# Split into train/validation
train_files, val_files = train_test_split(
    [f[0] for f in train_data],
    train_size=TRAIN_RATIO,
    random_state=42
)

# Move validation files
for file in val_files:
    shutil.move(f'data/train/images/{file}', f'data/val/images/{file}')
    shutil.move(f'data/train/labels/{file.replace(".png", ".txt")}',
               f'data/val/labels/{file.replace(".png", ".txt")}')

# Create data.yaml
data_yaml = f"""
train: {os.path.abspath('data/train/images')}
val: {os.path.abspath('data/val/images')}
nc: 1
names: ['building']
"""

with open('data.yaml', 'w') as f:
    f.write(data_yaml)

# Visualize sample tile and mask
sample_idx = 0
sample_img = cv2.imread(f'data/train/images/{train_files[sample_idx]}')
sample_mask = np.zeros_like(sample_img[:,:,0])

with open(f'data/train/labels/{train_files[sample_idx].replace(".png", ".txt")}', 'r') as f:
    for line in f:
        data = list(map(float, line.strip().split()[1:]))
        points = np.array(data).reshape(-1, 2)
        points[:, 0] *= sample_img.shape[1]  # x * width
        points[:, 1] *= sample_img.shape[0]  # y * height
        points = points.astype(np.int32)
        cv2.fillPoly(sample_mask, [points.reshape(-1, 1, 2)], 255)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
plt.title('Sample Tile')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(sample_mask, cmap='gray')
plt.title('Corresponding Mask')
plt.axis('off')
plt.tight_layout()
plt.savefig('sample_tile_mask.png', dpi=150, bbox_inches='tight')
plt.show()

# Prepare test image
print("Preparing test image...")
with rasterio.open('Test.tif') as src:
    test_img = src.read()
    test_transform = src.transform

test_tiles, test_coords = create_tiles(test_img, test_transform, TILE_SIZE, OVERLAP)

for i, tile in enumerate(test_tiles):
    tile_img = np.moveaxis(tile, 0, -1)
    cv2.imwrite(f'data/test/images/test_tile_{i:04d}.png', tile_img)

# Training function with memory management
def train_yolo_segmentation():
    """Train YOLOv8 segmentation with robust memory handling"""
    cleanup_memory()

    # Initialize model
    model = YOLO(MODEL_VARIANT)

    # Training parameters
    train_params = {
        'data': 'data.yaml',
        'epochs': EPOCHS,
        'imgsz': TILE_SIZE,
        'batch': BATCH_SIZE,
        'amp': True,  # Mixed precision
        'patience': 10,
        'save': True,
        'save_period': 5,
        'project': 'building_segmentation',
        'name': 'train_run',
        'exist_ok': True
    }

    # Adaptive training with OOM handling
    current_batch = BATCH_SIZE
    success = False

    while not success and current_batch >= 1:
        try:
            train_params['batch'] = current_batch
            print(f"Training with batch size: {current_batch}")

            # Train model
            results = model.train(**train_params)
            success = True

        except RuntimeError as e:
            if 'CUDA out of memory' in str(e):
                print(f"OOM with batch size {current_batch}, reducing...")
                current_batch = max(1, current_batch // 2)
                cleanup_memory()
            else:
                raise e
        except Exception as e:
            print(f"Training error: {e}")
            break

    if success:
        print("Training completed successfully!")
        return model, results
    else:
        print("Training failed after reducing batch size")
        return None, None

# Train the model
print("Starting training...")
model, results = train_yolo_segmentation()

if model is not None:
    # Load best model
    best_model_path = 'building_segmentation/train_run/weights/best.pt'
    if os.path.exists(best_model_path):
        model = YOLO(best_model_path)

    # Plot training results
    results.plot()
    plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Test on sample tiles
    print("Running predictions on test tiles...")
    test_files = sorted(os.listdir('data/test/images'))[:5]  # Test on first 5 tiles

    for test_file in test_files:
        test_path = f'data/test/images/{test_file}'
        results = model(test_path, conf=0.3, imgsz=TILE_SIZE)

        # Visualize results
        for result in results:
            result.save(filename=f'test_predictions/{test_file}')

            # Plot
            plt.figure(figsize=(10, 5))
            plt.subplot(1, 2, 1)
            plt.imshow(cv2.cvtColor(cv2.imread(test_path), cv2.COLOR_BGR2RGB))
            plt.title('Original')
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(cv2.cvtColor(cv2.imread(f'test_predictions/{test_file}'), cv2.COLOR_BGR2RGB))
            plt.title('Prediction')
            plt.axis('off')

            plt.tight_layout()
            plt.savefig(f'test_predictions/compare_{test_file}', dpi=150, bbox_inches='tight')
            plt.show()

    print("Pipeline completed successfully!")
    print(f"Best model saved at: {best_model_path}")
    print(f"Training results saved as: training_results.png")
    print(f"Sample predictions saved in: test_predictions/")

else:
    print("Training failed. Consider reducing TILE_SIZE or using a smaller model variant.")

# Final cleanup
cleanup_memory()
print("Process completed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 29.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Please upload your files:
1. Train.tif - Training aerial imagery
2. Test.tif - Test aerial imagery
3. Shapefile components (.shp, .shx, .dbf, .prj)
